In [1]:
# ============================================
# 1. IMPORTS
# ============================================

from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    precision_recall_curve
)

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

print("All imports successful.")

All imports successful.


In [2]:
# ============================================
# 2. PROJECT PATHS
# ============================================

from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"

MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"
TABLES_DIR = REPORTS_DIR / "tables"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

ML_DATASET_PATH = PROCESSED_DIR / "ml_dataset.parquet"

print("Project root:", PROJECT_ROOT)
print("ML dataset:", ML_DATASET_PATH)

Project root: D:\01_PROJECTS\DS_AI\flood-prediction-india
ML dataset: D:\01_PROJECTS\DS_AI\flood-prediction-india\data\processed\ml_dataset.parquet


In [3]:
# ============================================
# 3. VERIFY DATASET
# ============================================

print("Dataset exists:", ML_DATASET_PATH.exists())

if ML_DATASET_PATH.exists():
    print(
        f"Dataset size: "
        f"{ML_DATASET_PATH.stat().st_size / (1024**2):.2f} MB"
    )

Dataset exists: True
Dataset size: 145.31 MB


In [4]:
# ============================================
# 4. LOAD ML DATASET
# ============================================

ml_dataset = pd.read_parquet(ML_DATASET_PATH)

print("Shape:", ml_dataset.shape)
print("\nColumns:")
print(ml_dataset.columns.tolist())

Shape: (3292158, 20)

Columns:
['station_code', 'date', 'water_level', 'water_level_lag_1', 'water_level_lag_2', 'water_level_lag_3', 'level_change_1d', 'level_change_3d', 'rain_1d', 'rain_3d', 'rain_7d', 'level_danger_ratio', 'level_warning_ratio', 'rise_danger_ratio_1d', 'rise_danger_ratio_3d', 'danger_level', 'warning_level', 'lat', 'lon', 'flood_next_day']


In [5]:
# ============================================
# 5. ENSURE DATE TYPE
# ============================================

ml_dataset["date"] = pd.to_datetime(ml_dataset["date"])

print("Date dtype:", ml_dataset["date"].dtype)
print("Date range:")
print(ml_dataset["date"].min(), "→", ml_dataset["date"].max())

Date dtype: datetime64[us]
Date range:
1964-06-19 00:00:00 → 2026-09-10 00:00:00


In [6]:
# ============================================
# 6. CHRONOLOGICAL TRAIN / VALIDATION / TEST SPLIT
# ============================================

train_df = ml_dataset[
    ml_dataset["date"].dt.year <= 2022
].copy()

val_df = ml_dataset[
    ml_dataset["date"].dt.year.isin([2023, 2024])
].copy()

test_df = ml_dataset[
    ml_dataset["date"].dt.year >= 2025
].copy()

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (2471787, 20)
Validation: (446049, 20)
Test: (374322, 20)


In [7]:
# ============================================
# 7. VERIFY SPLIT
# ============================================

for name, df in [
    ("TRAIN", train_df),
    ("VALIDATION", val_df),
    ("TEST", test_df)
]:
    print(f"\n{name}")
    print("-" * 40)
    print("Rows:", len(df))
    print("Date:", df["date"].min(), "→", df["date"].max())
    print("Flood events:", int(df["flood_next_day"].sum()))
    print("Flood rate:", df["flood_next_day"].mean())


TRAIN
----------------------------------------
Rows: 2471787
Date: 1964-06-19 00:00:00 → 2022-12-31 00:00:00
Flood events: 45862
Flood rate: 0.01855418771924927

VALIDATION
----------------------------------------
Rows: 446049
Date: 2023-01-01 00:00:00 → 2024-12-31 00:00:00
Flood events: 4773
Flood rate: 0.01070061809352784

TEST
----------------------------------------
Rows: 374322
Date: 2025-01-01 00:00:00 → 2026-09-10 00:00:00
Flood events: 4279
Flood rate: 0.011431334519477882


In [8]:
# ============================================
# 8. MODEL FEATURES
# ============================================

model_features = [
    "water_level",
    "water_level_lag_1",
    "water_level_lag_2",
    "water_level_lag_3",
    "level_change_1d",
    "level_change_3d",
    "rain_1d",
    "rain_3d",
    "rain_7d",
    "level_danger_ratio",
    "level_warning_ratio",
    "rise_danger_ratio_1d",
    "rise_danger_ratio_3d",
    "danger_level",
    "warning_level",
    "lat",
    "lon"
]

print("Number of features:", len(model_features))

for i, feature in enumerate(model_features, start=1):
    print(f"{i:2}. {feature}")

Number of features: 17
 1. water_level
 2. water_level_lag_1
 3. water_level_lag_2
 4. water_level_lag_3
 5. level_change_1d
 6. level_change_3d
 7. rain_1d
 8. rain_3d
 9. rain_7d
10. level_danger_ratio
11. level_warning_ratio
12. rise_danger_ratio_1d
13. rise_danger_ratio_3d
14. danger_level
15. warning_level
16. lat
17. lon


In [9]:
# ============================================
# 9. CREATE X / y MATRICES
# ============================================

X_train = train_df[model_features].copy()
y_train = train_df["flood_next_day"].astype(int)

X_val = val_df[model_features].copy()
y_val = val_df["flood_next_day"].astype(int)

X_test = test_df[model_features].copy()
y_test = test_df["flood_next_day"].astype(int)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (2471787, 17)
y_train: (2471787,)
X_val: (446049, 17)
y_val: (446049,)
X_test: (374322, 17)
y_test: (374322,)


In [10]:
train_df = ml_dataset[
    ml_dataset["date"].dt.year <= 2022
].copy()

val_df = ml_dataset[
    ml_dataset["date"].dt.year.isin([2023, 2024])
].copy()

test_df = ml_dataset[
    ml_dataset["date"].dt.year >= 2025
].copy()

In [11]:
print("TRAIN")
print("Rows:", len(train_df))
print(
    "Date:",
    train_df["date"].min(),
    "→",
    train_df["date"].max()
)
print(
    "Flood:",
    int(train_df["flood_next_day"].sum())
)
print(
    "Flood rate:",
    train_df["flood_next_day"].mean()
)

print("\nVALIDATION")
print("Rows:", len(val_df))
print(
    "Date:",
    val_df["date"].min(),
    "→",
    val_df["date"].max()
)
print(
    "Flood:",
    int(val_df["flood_next_day"].sum())
)
print(
    "Flood rate:",
    val_df["flood_next_day"].mean()
)

print("\nTEST")
print("Rows:", len(test_df))
print(
    "Date:",
    test_df["date"].min(),
    "→",
    test_df["date"].max()
)
print(
    "Flood:",
    int(test_df["flood_next_day"].sum())
)
print(
    "Flood rate:",
    test_df["flood_next_day"].mean()
)

TRAIN
Rows: 2471787
Date: 1964-06-19 00:00:00 → 2022-12-31 00:00:00
Flood: 45862
Flood rate: 0.01855418771924927

VALIDATION
Rows: 446049
Date: 2023-01-01 00:00:00 → 2024-12-31 00:00:00
Flood: 4773
Flood rate: 0.01070061809352784

TEST
Rows: 374322
Date: 2025-01-01 00:00:00 → 2026-09-10 00:00:00
Flood: 4279
Flood rate: 0.011431334519477882


In [12]:
print(
    "Train/Validation overlap:",
    len(
        set(train_df["date"])
        & set(val_df["date"])
    )
)

print(
    "Validation/Test overlap:",
    len(
        set(val_df["date"])
        & set(test_df["date"])
    )
)

print(
    "Train/Test overlap:",
    len(
        set(train_df["date"])
        & set(test_df["date"])
    )
)

Train/Validation overlap: 0
Validation/Test overlap: 0
Train/Test overlap: 0


In [13]:
model_features = [
    "water_level",
    "water_level_lag_1",
    "water_level_lag_2",
    "water_level_lag_3",
    "level_change_1d",
    "level_change_3d",
    "rain_1d",
    "rain_3d",
    "rain_7d",
    "level_danger_ratio",
    "level_warning_ratio",
    "rise_danger_ratio_1d",
    "rise_danger_ratio_3d",
    "danger_level",
    "warning_level",
    "lat",
    "lon"
]

In [14]:
X_train = train_df[model_features].copy()
y_train = train_df["flood_next_day"].astype(int)

X_val = val_df[model_features].copy()
y_val = val_df["flood_next_day"].astype(int)

X_test = test_df[model_features].copy()
y_test = test_df["flood_next_day"].astype(int)

In [15]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

print("\nTraining target:")
print(y_train.value_counts())

print("\nValidation target:")
print(y_val.value_counts())

print("\nTest target:")
print(y_test.value_counts())

X_train: (2471787, 17)
y_train: (2471787,)
X_val: (446049, 17)
y_val: (446049,)
X_test: (374322, 17)
y_test: (374322,)

Training target:
flood_next_day
0    2425925
1      45862
Name: count, dtype: int64

Validation target:
flood_next_day
0    441276
1      4773
Name: count, dtype: int64

Test target:
flood_next_day
0    370043
1      4279
Name: count, dtype: int64


In [16]:
print("Number of features:", len(model_features))
print("\nFeatures:")
for i, feature in enumerate(model_features, start=1):
    print(f"{i:2}. {feature}")

Number of features: 17

Features:
 1. water_level
 2. water_level_lag_1
 3. water_level_lag_2
 4. water_level_lag_3
 5. level_change_1d
 6. level_change_3d
 7. rain_1d
 8. rain_3d
 9. rain_7d
10. level_danger_ratio
11. level_warning_ratio
12. rise_danger_ratio_1d
13. rise_danger_ratio_3d
14. danger_level
15. warning_level
16. lat
17. lon


In [17]:
print("\nX_train columns:")
print(X_train.columns.tolist())


X_train columns:
['water_level', 'water_level_lag_1', 'water_level_lag_2', 'water_level_lag_3', 'level_change_1d', 'level_change_3d', 'rain_1d', 'rain_3d', 'rain_7d', 'level_danger_ratio', 'level_warning_ratio', 'rise_danger_ratio_1d', 'rise_danger_ratio_3d', 'danger_level', 'warning_level', 'lat', 'lon']


In [18]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

print(logistic_model)

Pipeline(steps=[('scaler', StandardScaler()),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=1000,
                                    random_state=42))])


In [19]:
import time

start_time = time.time()

logistic_model.fit(X_train, y_train)

training_time = time.time() - start_time

print(f"Logistic Regression training completed.")
print(f"Training time: {training_time:.2f} seconds")

Logistic Regression training completed.
Training time: 11.31 seconds


In [20]:
logistic_estimator = logistic_model.named_steps["model"]

print("Converged iterations:", logistic_estimator.n_iter_)
print("Maximum iterations:", logistic_estimator.max_iter)
print("Number of coefficients:", logistic_estimator.coef_.shape)
print("Intercept:", logistic_estimator.intercept_)

Converged iterations: [80]
Maximum iterations: 1000
Number of coefficients: (1, 17)
Intercept: [-7.50920153]


In [21]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

# Predict probability of flood
val_prob = logistic_model.predict_proba(X_val)[:, 1]

# Default threshold = 0.5
val_pred = (val_prob >= 0.5).astype(int)

print("Validation ROC-AUC:", roc_auc_score(y_val, val_prob))
print("Validation PR-AUC:", average_precision_score(y_val, val_prob))
print("Validation Precision:", precision_score(y_val, val_pred, zero_division=0))
print("Validation Recall:", recall_score(y_val, val_pred, zero_division=0))
print("Validation F1:", f1_score(y_val, val_pred, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, val_pred))

Validation ROC-AUC: 0.9872483123418783
Validation PR-AUC: 0.725539776104448
Validation Precision: 0.30533689126084057
Validation Recall: 0.9589356798659124
Validation F1: 0.4631887871274604

Confusion Matrix:
[[430863  10413]
 [   196   4577]]


In [22]:
from sklearn.metrics import precision_recall_curve
import numpy as np
import pandas as pd

precision, recall, thresholds = precision_recall_curve(
    y_val,
    val_prob
)

threshold_results = pd.DataFrame({
    "threshold": thresholds,
    "precision": precision[:-1],
    "recall": recall[:-1]
})

threshold_results["f1"] = (
    2 * threshold_results["precision"] * threshold_results["recall"]
    / (
        threshold_results["precision"]
        + threshold_results["recall"]
    )
)

# Show some useful threshold points
selected_thresholds = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]

print(
    threshold_results[
        threshold_results["threshold"].isin(selected_thresholds)
    ].to_string(index=False)
)

Empty DataFrame
Columns: [threshold, precision, recall, f1]
Index: []


In [23]:
best_f1_row = threshold_results.loc[
    threshold_results["f1"].idxmax()
]

print("Best validation F1 threshold:")
print(best_f1_row)

Best validation F1 threshold:
threshold    0.739413
precision    0.539363
recall       0.887073
f1           0.670839
Name: 401649, dtype: float64


In [24]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds_to_check = [
    0.30,
    0.40,
    0.50,
    0.60,
    0.70,
    0.739413,
    0.80,
    0.90
]

rows = []

for threshold in thresholds_to_check:
    pred = (val_prob >= threshold).astype(int)

    rows.append({
        "threshold": threshold,
        "precision": precision_score(y_val, pred, zero_division=0),
        "recall": recall_score(y_val, pred, zero_division=0),
        "f1": f1_score(y_val, pred, zero_division=0),
        "predicted_floods": pred.sum()
    })

threshold_table = pd.DataFrame(rows)

print(threshold_table.to_string(index=False))

 threshold  precision   recall       f1  predicted_floods
  0.300000   0.144421 0.978630 0.251697             32343
  0.400000   0.223832 0.974021 0.364014             20770
  0.500000   0.305337 0.958936 0.463189             14990
  0.600000   0.400909 0.942175 0.562477             11217
  0.700000   0.497874 0.907605 0.643016              8701
  0.739413   0.539304 0.886864 0.670734              7849
  0.800000   0.571358 0.726378 0.639609              6068
  0.900000   0.696390 0.602137 0.645843              4127


In [25]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_leaf=5,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

print(rf_model)

RandomForestClassifier(class_weight='balanced', max_depth=20,
                       min_samples_leaf=5, n_estimators=200, n_jobs=-1,
                       random_state=42)


In [26]:
import time

start_time = time.time()

rf_model.fit(X_train, y_train)

rf_training_time = time.time() - start_time

print("Random Forest training completed.")
print(f"Training time: {rf_training_time:.2f} seconds")

Random Forest training completed.
Training time: 202.44 seconds


In [27]:
import time

start_time = time.time()

rf_val_prob = rf_model.predict_proba(X_val)[:, 1]

rf_prediction_time = time.time() - start_time

print(f"Validation prediction time: {rf_prediction_time:.2f} seconds")
print(f"Probability shape: {rf_val_prob.shape}")
print(f"Minimum probability: {rf_val_prob.min():.6f}")
print(f"Maximum probability: {rf_val_prob.max():.6f}")

Validation prediction time: 0.78 seconds
Probability shape: (446049,)
Minimum probability: 0.000000
Maximum probability: 0.999961


In [28]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

rf_val_pred = (rf_val_prob >= 0.5).astype(int)

print("Random Forest — Validation Results")
print("----------------------------------")
print(f"ROC-AUC:  {roc_auc_score(y_val, rf_val_prob):.6f}")
print(f"PR-AUC:   {average_precision_score(y_val, rf_val_prob):.6f}")
print(f"Precision:{precision_score(y_val, rf_val_pred, zero_division=0):.6f}")
print(f"Recall:   {recall_score(y_val, rf_val_pred, zero_division=0):.6f}")
print(f"F1:       {f1_score(y_val, rf_val_pred, zero_division=0):.6f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, rf_val_pred))

Random Forest — Validation Results
----------------------------------
ROC-AUC:  0.994601
PR-AUC:   0.936750
Precision:0.736746
Recall:   0.931699
F1:       0.822833

Confusion Matrix:
[[439687   1589]
 [   326   4447]]


In [29]:
from sklearn.metrics import precision_recall_curve

rf_precision, rf_recall, rf_thresholds = precision_recall_curve(
    y_val,
    rf_val_prob
)

rf_threshold_results = pd.DataFrame({
    "threshold": rf_thresholds,
    "precision": rf_precision[:-1],
    "recall": rf_recall[:-1]
})

rf_threshold_results["f1"] = (
    2 * rf_threshold_results["precision"] * rf_threshold_results["recall"]
    / (
        rf_threshold_results["precision"]
        + rf_threshold_results["recall"]
    )
)

best_rf_f1 = rf_threshold_results.loc[
    rf_threshold_results["f1"].idxmax()
]

print("Best Random Forest validation F1 threshold:")
print(best_rf_f1)

Best Random Forest validation F1 threshold:
threshold    0.904376
precision    0.908869
recall       0.856694
f1           0.882010
Name: 301209, dtype: float64


In [30]:
thresholds_to_check = [
    0.30,
    0.40,
    0.50,
    0.60,
    0.70,
    0.80,
    0.90
]

rows = []

for threshold in thresholds_to_check:
    pred = (rf_val_prob >= threshold).astype(int)

    rows.append({
        "threshold": threshold,
        "precision": precision_score(y_val, pred, zero_division=0),
        "recall": recall_score(y_val, pred, zero_division=0),
        "f1": f1_score(y_val, pred, zero_division=0),
        "predicted_floods": pred.sum()
    })

rf_threshold_table = pd.DataFrame(rows)

print(rf_threshold_table.to_string(index=False))

 threshold  precision   recall       f1  predicted_floods
       0.3   0.624295 0.950765 0.753695              7269
       0.4   0.687500 0.942594 0.795087              6544
       0.5   0.736746 0.931699 0.822833              6036
       0.6   0.775388 0.921433 0.842125              5672
       0.7   0.818698 0.906348 0.860296              5284
       0.8   0.860801 0.887492 0.873943              4921
       0.9   0.905731 0.857532 0.880973              4519


In [31]:
scale_pos_weight = (
    y_train.value_counts()[0]
    / y_train.value_counts()[1]
)

print(f"Negative samples: {y_train.value_counts()[0]:,}")
print(f"Positive samples: {y_train.value_counts()[1]:,}")
print(f"scale_pos_weight: {scale_pos_weight:.6f}")

Negative samples: 2,425,925
Positive samples: 45,862
scale_pos_weight: 52.896189


In [32]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    n_jobs=-1,
    random_state=42
)

print(xgb_model)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric='aucpr', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=8,
              max_leaves=None, min_child_weight=5, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=500,
              n_jobs=-1, num_parallel_tree=None, ...)


In [33]:
import time

start_time = time.time()

xgb_model.fit(
    X_train,
    y_train
)

xgb_training_time = time.time() - start_time

print("XGBoost training completed.")
print(f"Training time: {xgb_training_time:.2f} seconds")

XGBoost training completed.
Training time: 38.07 seconds


In [34]:
import time

start_time = time.time()

xgb_val_prob = xgb_model.predict_proba(X_val)[:, 1]

xgb_prediction_time = time.time() - start_time

print(f"Validation prediction time: {xgb_prediction_time:.2f} seconds")
print(f"Probability shape: {xgb_val_prob.shape}")
print(f"Minimum probability: {xgb_val_prob.min():.6f}")
print(f"Maximum probability: {xgb_val_prob.max():.6f}")

Validation prediction time: 0.54 seconds
Probability shape: (446049,)
Minimum probability: 0.000007
Maximum probability: 0.999950


In [35]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

xgb_val_pred = (xgb_val_prob >= 0.5).astype(int)

print("XGBoost — Validation Results")
print("----------------------------")
print(f"ROC-AUC:  {roc_auc_score(y_val, xgb_val_prob):.6f}")
print(f"PR-AUC:   {average_precision_score(y_val, xgb_val_prob):.6f}")
print(f"Precision:{precision_score(y_val, xgb_val_pred, zero_division=0):.6f}")
print(f"Recall:   {recall_score(y_val, xgb_val_pred, zero_division=0):.6f}")
print(f"F1:       {f1_score(y_val, xgb_val_pred, zero_division=0):.6f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, xgb_val_pred))

XGBoost — Validation Results
----------------------------
ROC-AUC:  0.994869
PR-AUC:   0.938235
Precision:0.630311
Recall:   0.954117
F1:       0.759127

Confusion Matrix:
[[438605   2671]
 [   219   4554]]


In [36]:
xgb_precision, xgb_recall, xgb_thresholds = precision_recall_curve(
    y_val,
    xgb_val_prob
)

xgb_threshold_results = pd.DataFrame({
    "threshold": xgb_thresholds,
    "precision": xgb_precision[:-1],
    "recall": xgb_recall[:-1]
})

xgb_threshold_results["f1"] = (
    2 * xgb_threshold_results["precision"] * xgb_threshold_results["recall"]
    / (
        xgb_threshold_results["precision"]
        + xgb_threshold_results["recall"]
    )
)

best_xgb_f1 = xgb_threshold_results.loc[
    xgb_threshold_results["f1"].idxmax()
]

print("Best XGBoost validation F1 threshold:")
print(best_xgb_f1)

Best XGBoost validation F1 threshold:
threshold    0.955972
precision    0.890060
recall       0.866750
f1           0.878251
Name: 370864, dtype: float64


In [37]:
thresholds_to_check = [
    0.30,
    0.40,
    0.50,
    0.60,
    0.70,
    0.80,
    0.90
]

rows = []

for threshold in thresholds_to_check:
    pred = (xgb_val_prob >= threshold).astype(int)

    rows.append({
        "threshold": threshold,
        "precision": precision_score(y_val, pred, zero_division=0),
        "recall": recall_score(y_val, pred, zero_division=0),
        "f1": f1_score(y_val, pred, zero_division=0),
        "predicted_floods": pred.sum()
    })

xgb_threshold_table = pd.DataFrame(rows)

print(xgb_threshold_table.to_string(index=False))

 threshold  precision   recall       f1  predicted_floods
       0.3   0.523257 0.963964 0.678313              8793
       0.4   0.581723 0.958936 0.724152              7868
       0.5   0.630311 0.954117 0.759127              7225
       0.6   0.674332 0.946155 0.787446              6697
       0.7   0.720122 0.937984 0.814741              6217
       0.8   0.768776 0.926461 0.840285              5752
       0.9   0.834592 0.902787 0.867351              5163


In [38]:
import joblib

joblib.dump(
    logistic_model,
    MODELS_DIR / "logistic_regression.joblib"
)

['D:\\01_PROJECTS\\DS_AI\\flood-prediction-india\\models\\logistic_regression.joblib']

In [39]:
joblib.dump(
    rf_model,
    MODELS_DIR / "random_forest.joblib"
)

['D:\\01_PROJECTS\\DS_AI\\flood-prediction-india\\models\\random_forest.joblib']

In [40]:
joblib.dump(
    xgb_model,
    MODELS_DIR / "xgboost.joblib"
)

['D:\\01_PROJECTS\\DS_AI\\flood-prediction-india\\models\\xgboost.joblib']